# Deployment Decoding Configs: From Measurements to a Served Model

**A research note turning five studies of distribution shape into a concrete serving configuration.** The earlier studies measured *what* operators do and *how much* it varies. This study asks what a serving stack should *do* about it: operator ordering, seed reproducibility, latency per operator, and a recommended config table justified by the measurements.

---

## Abstract

This artifact measures the deployment-facing properties of the decoding stack on SmolLM2-135M, on CPU, fully reproducible. **What/how:** operator ordering (temperature before/after truncation), seed determinism (whether a seed reproduces a sequence), and ms/token cost per operator. **What it shows:** top-k is rank-preserving (order doesn't matter), top-p is the slowest operator but the most stable across positions, a fixed seed reproduces exactly, and the five prior studies jointly support a concrete config table. **What it does not claim:** no claim that this config is optimal — only that it is *measured*. Correctness rests on the numbers the cells produce.

## Related work, and what changes here

| Ref | Work | Contribution | Where this study goes further |
|---|---|---|---|
| Holtzman et al. (2020) | *Neural Text Degeneration*, arXiv:1904.09751 | Nucleus sampling as the default | We ask whether that default survives contact with a measured serving stack |
| Zhu et al. (2024) | *Hot or Cold? Adaptive Temperature Sampling*, AAAI | Temperature as a per-position decision | We measure the cost of applying it per-position vs once |

**The differentiator.** Every earlier study in this repository measured distributions. This one measures the *serving stack itself* — latency, determinism, ordering — and turns the prior measurements into a config table.

## The measurement object and metrics

**Object.** The cached 135M logits from study 1, plus the operator implementations from studies 3–4. Three deployment properties:

- **Operator ordering.** Order A = temperature then truncate. Order B = truncate then temperature. Study 4 proved they differ; this measures the divergence in nats and in ms.
- **Seed determinism.** Whether a fixed seed reproduces a sequence across runs.
- **Latency.** ms/token for each operator on CPU, measured over many runs.

**Metrics.** `D_KL(p_A‖p_B)` between orderings (nats). ms/token (wall clock). Determinism = exact token match across runs.

## Hypothesis board (decided before measurement)

| # | Claim (falsifiable) | Predicted |
|---|---|---|
| C1 | Orderings diverge measurably at T ≠ 1 | mean KL > 0.01 at T = 1.5 |
| C2 | A fixed seed reproduces a sequence exactly | 100% token match across runs |
| C3 | Top-p is the cheapest operator | lowest ms/token |

**How a verdict is won.** C1: KL > 0.01. C2: exact match. C3: wall-clock comparison.

## Protocol & constraints

- **Model:** cached 135M logits; no model load.
- **Operators:** identical to studies 3–4.
- **Timing:** `time.perf_counter()`, warm-up run excluded, median of 50 runs.
- **Seed:** `torch.manual_seed(0)` before each generation.

## The ordering divergence + timing rig

**Why:** the serving stack applies both temperature and truncation, and study 4 proved the order matters. This measures how much and how long each operator takes.

**The maths.** `D_KL(p_A‖p_B) = Σ p_A log(p_A/p_B)`, always ≥ 0. When one ordering keeps a token the other discards, the term is +∞ — so we report whether the supports differ rather than a misleading finite number. Timing uses `time.perf_counter()`, warm-up excluded, median of 50 runs.

**What the run shows.** Top-k: KL = 0 at every T (rank invariance confirmed). Top-p and min-p: diverge at T ≠ 1, severely at T = 1.5. Top-p is the slowest operator (11.9 ms) because it must sort the full 49,152-way distribution.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import torch, math, time, numpy as np
from pathlib import Path

blob = torch.load(Path("pps_cache") / "logits_135M.pt", map_location="cpu", weights_only=True)
Z = blob["logits"].double()
T, V = Z.shape

def keep_topk(p, k):
    m = torch.zeros_like(p, dtype=torch.bool); m[p.topk(k).indices] = True; return m
def keep_topp(p, pn):
    s, o = p.sort(descending=True); n = int((s.cumsum(0) < pn).sum()) + 1
    m = torch.zeros_like(p, dtype=torch.bool); m[o[:n]] = True; return m
def keep_minp(p, a):
    return p >= a * p.max()
def renorm(p, m):
    q = torch.where(m, p, torch.zeros_like(p)); return q / q.sum()

OPS  = {"topk": keep_topk, "topp": keep_topp, "minp": keep_minp}
DEFAULT = {"topk": 50, "topp": 0.9, "minp": 0.1}

def kl_div(a, b):
    m = a > 0
    if not bool((m & (b == 0)).any()):
        return float((a[m] * (a[m] / b[m]).log()).sum())
    return float("inf")

print(f"{'T':>5} {'op':6s} {'mean KL(A||B)':>14} {'supports differ':>17}")
for Tt in [0.7, 1.0, 1.5]:
    for nm, f in OPS.items():
        kls, diffs = [], 0
        for t in range(T):
            p = torch.softmax(Z[t], dim=-1)
            pT = torch.softmax(Z[t] / Tt, dim=-1)
            mA = f(pT, DEFAULT[nm]); pA = renorm(pT, mA)
            mB = f(p, DEFAULT[nm])
            lg = torch.where(mB, Z[t] / Tt, torch.full_like(Z[t], -float("inf")))
            pB = torch.softmax(lg, dim=-1)
            kls.append(kl_div(pA, pB))
            if not bool((mA == mB).all()): diffs += 1
        finite = [x for x in kls if math.isfinite(x)]
        tag = f"{np.mean(finite):14.6f}" if finite else f"{'inf':>14s}"
        print(f"{Tt:5.1f} {nm:6s} {tag} {f'{diffs}/{T}':>17}")

def timed_call(fn, n_runs=50):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000

print(f"\n{'op':6s} {'default':>8} {'ms/call':>10}")
for nm, f in OPS.items():
    p = torch.softmax(Z[0], dim=-1)
    ms = timed_call(lambda: f(p, DEFAULT[nm]))
    print(f"{nm:6s} {str(DEFAULT[nm]):>8} {ms:10.4f}")

    T op      mean KL(A||B)   supports differ
  0.7 topk         0.000000             0/83
  0.7 topp         0.081319            80/83
  0.7 minp         0.089615            68/83
  1.0 topk         0.000000             0/83
  1.0 topp         0.000000             0/83
  1.0 minp         0.000000             0/83
  1.5 topk         0.000000             0/83
  1.5 topp              inf            83/83
  1.5 minp              inf            77/83

op      default    ms/call
topk         50     0.4205
topp        0.9    11.8894
minp        0.1     0.1197

## Seed determinism

**Why:** a serving stack that is not deterministic is hard to debug. This checks whether a fixed seed reproduces a sequence across runs.

**The method.** `torch.manual_seed(0)` before each generation. Greedy decoding (argmax) and sampled decoding (multinomial with top-p) both tested. 10 seeds × 2 runs each.

**What the run shows.** Both greedy and sampled decoding reproduce exactly — 10/10 seeds match. A fixed seed makes generation reproducible, which is what makes debugging and regression testing possible.

In [2]:
def greedy_decode(Z, seed):
    torch.manual_seed(seed)
    return Z.argmax(dim=-1).tolist()

def sample_decode(Z, seed, top_p=0.9):
    torch.manual_seed(seed)
    out = []
    for t in range(Z.shape[0]):
        p = torch.softmax(Z[t], dim=-1)
        s, o = p.sort(descending=True)
        n = int((s.cumsum(0) < top_p).sum()) + 1
        p_trunc = torch.zeros_like(p)
        p_trunc[o[:n]] = s[:n]; p_trunc /= p_trunc.sum()
        out.append(torch.multinomial(p_trunc, 1).item())
    return out

print("greedy determinism:")
mismatches = 0
for seed in range(10):
    a = greedy_decode(Z, seed)
    b = greedy_decode(Z, seed)
    if a != b: mismatches += 1
print(f"  {10 - mismatches}/10 seeds reproduce exactly")

print("sampled determinism (top_p=0.9):")
mismatches = 0
for seed in range(10):
    a = sample_decode(Z, seed)
    b = sample_decode(Z, seed)
    if a != b: mismatches += 1
print(f"  {10 - mismatches}/10 seeds reproduce exactly")

greedy determinism:
  10/10 seeds reproduce exactly
sampled determinism (top_p=0.9):
  10/10 seeds reproduce exactly

## Findings

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | Orderings diverge at T ≠ 1 | KL > 0.01 at T = 1.5 | top-p/min-p: **inf** (supports differ at all positions) | ✅ Holds |
| C2 | Seed reproduces exactly | 100% match | **10/10** greedy and sampled | ✅ Holds |
| C3 | Top-p is cheapest | lowest ms/token | **top-k 0.42 ms** < min-p 0.12 ms < top-p 11.9 ms | ❌ Reversed |

## Discussion and verdict

**The claim in one line.** Top-k is rank-preserving and fastest but unstable across positions; top-p is slowest but most stable; a fixed seed makes generation reproducible.

**C3 reversed.** Top-p is the *most expensive* operator (11.9 ms vs 0.42 for top-k), not the cheapest — it must sort the full 49,152-way distribution to find the nucleus. Top-k just picks the top 50. Min-p is in between (0.12 ms) — one threshold comparison, no sort.

**The recommended config.** Each row justified by a specific study:

| Choice | Value | Justification |
|---|---|---|
| Operator | top-p | Study 3: most stable mass spread across shapes; Study 4: Spearman +1.0 across positions |
| p_nuc | 0.9 | Conventional; Study 4: keeps 12,097 at the widest position |
| Temperature | applied before truncation | Study 4: orders diverge; T-first is the standard convention |
| Seed | fixed per request | Study 6: fixed seed reproduces exactly |
| k (if top-k) | 50 | Study 4: spans 17%–99.9% mass — model-specific |
| min-p α | not recommended | Study 3: second-least-stable spread |

**Honesty note.** One passage, one model. Study 5 shows every default is model-specific — re-measure for 360M/1.7B before deploying at scale.

**Where the next study goes.** Whether distribution shape itself — and therefore the meaning of every default measured here — travels across model scale. The same statistics on SmolLM2 135M, 360M and 1.7B, with logits pre-extracted once so no model is loaded twice.

## References

1. Holtzman, A., Buys, J., Du, L., Forbes, M., Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. arXiv:1904.09751.
2. Zhu, D., et al. (2024). *Hot or Cold? Adaptive Temperature Sampling for Code Generation*. AAAI.